[Homework: Agentic RAG]('https://colab.research.google.com/github/Jaguar838/llm-zoomcamp/blob/main/HW/2026/01-agentic-rag/hw-01.ipynb')

In this homework, we build a RAG system from scratch and then make it
agentic - the same path as the module.

Instead of the course FAQ, our knowledge base is the course lessons
themselves.

The course repository is organized by module. Each module is a top-level
folder with a `lessons/` subfolder of numbered markdown pages:

```
01-agentic-rag/
└── lessons/
    ├── 01-intro.md
    ├── 02-environment.md
    ├── ...
    └── 16-other-frameworks.md
```

There are seven modules:

- `01-agentic-rag`
- `02-vector-search`
- `03-orchestration`
- `04-evaluation`
- `05-monitoring`
- `06-best-practices`
- `07-project-example`

Each lesson page is a single markdown file. These pages are exactly what you
read as you go through the course.

We'll fetch this data from GitHub and use it as the knowledge base for our
RAG system.

> It's possible your answers won't match exactly. If so, select the closest one.

## Setup

Prepare your environment the same way as in the module's
[Environment](../../../01-agentic-rag/lessons/02-environment.md) lesson.

This homework needs one extra library: `gitsource`, which downloads files
from a GitHub repository.

Install it:

```bash
uv add gitsource
```

For the LLM, we recommend OpenAI with `llama-3.3-70b-versatile`, but you can use any model
and provider you like - just adapt the client and the usage fields accordingly.

## Preparation

First, we will pull the lesson pages straight from the course repository.
We will use the commit `8c1834d` to make sure everyone works with the exact same data.

We will use `gitsource` for that:

```python
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
```

`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks  the markdown files.

We also pass a `filename_filter` so we don't grab every markdown file in the
repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so
filtering on `/lessons/` keeps just those.


Each file has a `parse()` method that returns a dictionary with its
`filename` and `content`:

```python
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
```

In [ ]:
# Install necessary Python packages for data processing, search, LLM interaction, environment variable management, and Git data reading.
%pip install requests minsearch groq gitsource sqlitesearch

In [2]:
def calculate_estimated_cost(input_tokens, output_tokens):
    # Example pricing for Groq's llama-3.3-70b-versatile (check Groq's official pricing for actual values)
    # Assuming: Input token price = $0.70 per 1M tokens, Output token price = $0.80 per 1M tokens
    price_per_million_input_tokens = 0.70
    price_per_million_output_tokens = 0.80

    # Calculate the estimated total cost based on token usage and example pricing.
    total_cost = (
        (input_tokens / 1_000_000) * price_per_million_input_tokens +
        (output_tokens / 1_000_000) * price_per_million_output_tokens
    )
    return total_cost

## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [3]:
from gitsource import GithubRepositoryDataReader

# Initialize GithubRepositoryDataReader to read markdown files from a specific GitHub repository commit.
# It filters for files in the 'lessons/' path.
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [4]:
# Read the files matching the criteria.
documents = [file.parse() for file in reader.read()]

# Display the first document to inspect its structure and content.
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
# Print the total number of lesson pages (documents) found.
print(f"Q1: {len(documents)}")

Q1: 72


## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [6]:
import minsearch

# Initialize minsearch.Index, specifying 'content' as a text field for full-text search
# and 'filename' as a keyword field for exact matching.
index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
# Fit the index with the loaded documents.
index.fit(documents)

In [7]:
# Define the search query for Q2.
query = "How does the agentic loop keep calling the model until it stops?"

In [8]:
# Perform a search on the index with the defined query and retrieve only the top 1 result.
result_search = index.search(query, num_results=1)

In [9]:
# Print the filename of the first result for Q2, as determined by manual inspection or prior execution.
print(f"length context: {len(result_search[0]['content'])} simbols")
print(f"Q2: {result_search[0]['filename']}")

length context: 10121 simbols
Q2: 01-agentic-rag/lessons/14-agentic-loop.md


In [10]:
import time
from sqlitesearch import TextSearchIndex

index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course.db"
)

for doc in documents:
    index_sql.add(doc)
    print(f"""Added: {doc["filename"]}...""")
    time.sleep(0.5)

index_sql.close()
print("Done. Index saved to lesson_llm_course.db")

Added: 01-agentic-rag/lessons/01-intro.md...
Added: 01-agentic-rag/lessons/02-environment.md...
Added: 01-agentic-rag/lessons/03-rag.md...
Added: 01-agentic-rag/lessons/04-dataset.md...
Added: 01-agentic-rag/lessons/05-search.md...
Added: 01-agentic-rag/lessons/06-building-prompt.md...
Added: 01-agentic-rag/lessons/07-llm.md...
Added: 01-agentic-rag/lessons/08-rag-helper.md...
Added: 01-agentic-rag/lessons/09-data-ingestion.md...
Added: 01-agentic-rag/lessons/10-rag-next-steps.md...
Added: 01-agentic-rag/lessons/11-agents-intro.md...
Added: 01-agentic-rag/lessons/12-rag-revision.md...
Added: 01-agentic-rag/lessons/13-function-calling.md...
Added: 01-agentic-rag/lessons/14-agentic-loop.md...
Added: 01-agentic-rag/lessons/15-frameworks.md...
Added: 01-agentic-rag/lessons/16-other-frameworks.md...
Added: 02-vector-search/lessons/01-intro.md...
Added: 02-vector-search/lessons/02-embeddings.md...
Added: 02-vector-search/lessons/03-embeddings-dataset.md...
Added: 02-vector-search/lessons/04-

In [11]:
index_sql.count()

72

In [12]:
import os

file_path = 'lesson_llm_course.db'
file_size_mb = os.path.getsize(file_path) / 1024 ** 2


print(f"File size of {file_path}: {file_size_mb:.2f} MB)")

File size of lesson_llm_course.db: 1.02 MB)


In [13]:
results = index_sql.search(query, num_results=72)
sql_result_search = [doc["filename"] for doc in results]

In [14]:
print(f"Count files: {len(sql_result_search)}")
print(f"Q2: {sql_result_search[0]}")

Count files: 59
Q2: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use `llama-3.3-70b-versatile`. How many input (prompt) tokens did we send to the model for
this request?

In [15]:
import os
from groq import Groq
from google.colab import userdata

api_key_groq = userdata.get('GROQ_API_KEY')

# Initialize the Groq client with the retrieved API key.
client_groq = Groq(api_key=api_key_groq)

In [16]:
# Send a chat completion request to the Groq API using the 'llama-3.1-8b-instant' model.
chat_completion = client_groq.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.1-8b-instant",
)

# Print the content of the assistant's response.
print(chat_completion.choices[0].message.content)

Fast language models have gained significant attention in recent years due to their vast potential in various areas of artificial intelligence and natural language processing (NLP). Here are some of the key importance of fast language models:

1. **Improved Efficiency**: Fast language models are designed to be more efficient than their slow counterparts, enabling them to process vast amounts of text data in shorter periods. This efficiency is crucial for applications where time is a constraint, such as real-time chatbots, sentiment analysis, and text summarization.

2. **Scalability**: As the size of language models grows, so does their computational requirements. Fast language models can scale up more easily to accommodate larger models and handle more complex tasks without compromising performance.

3. **Parallelization**: Fast language models are optimized for parallelization, which allows them to be run on multiple cores or even multiple machines simultaneously. This capability mak

In [17]:
# Define the system instructions for the RAG assistant.
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

# Define the prompt template that structures the question and context for the LLM.
PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [18]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-24 05:39:02--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-24 05:39:02 (32.9 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [19]:
from dataclasses import dataclass
from rag_helper import RAGBase as OriginalRAGBase

# Define a dataclass to store the result of a RAG query, including the answer and token counts.
@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int

# Define the RAGBase class by inheriting from OriginalRAGBase and adapting it for our schema.
class RAGBase(OriginalRAGBase):

    # self,
    # index,
    # llm_client,
    # instructions=INSTRUCTIONS,
    # prompt_template=PROMPT_TEMPLATE,

    # Method to perform a search using the provided index.
    # Adapted for the filename/content schema.
    def search(self, query, num_results=1):
        return self.index.search(query, num_results=num_results)

    # Method to build context string from search results.
    # Adapted for the filename/content schema.
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"Filename: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # Method to interact with the LLM.
    # Overridden to use chat.completions.create and return the full response object for token counting.
    def llm(self, prompt):
        input_messages = [
            {"role": "system", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response

    # Main RAG method to perform the entire RAG process.
    # Overridden to return a RAGResult with token counts.
    def rag(self, query):
        search_results = self.search(query)
        # print(search_results)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)


        # Extract the answer and token usage from the LLM response.
        answer = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        return RAGResult(answer=answer, input_tokens=input_tokens, output_tokens=output_tokens)

In [20]:
# Initialize RAGBase with the document index, Groq client, and specified model.
rag = RAGBase(
    index=index_sql,
    llm_client=client_groq,
    model="llama-3.3-70b-versatile"
)
# Run the RAG query.
result = rag.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q3.
Q3_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q3_input_tokens)

The agentic loop keeps calling the model until it stops by using a `while` loop that continues to iterate as long as the model returns a response with function calls. The loop checks for the presence of function calls in the model's response using the `has_function_calls` flag. If the response contains a function call, the flag is set to `True`, and the loop continues. If the response does not contain a function call, the flag is set to `False`, and the loop exits.

The relevant code snippet is:
```python
while True:
    ...
    if has_function_calls == False:
        break
```
This loop will keep calling the model until it returns a response without any function calls, at which point it will stop.
Input tokens: 2399


In [21]:
# Print the input and output tokens for Q3.
print(f"""Q3:
input tokens: {Q3_input_tokens}
output tokens: {result.output_tokens}""")

estimated_cost = calculate_estimated_cost(Q3_input_tokens, result.output_tokens)
print(f"Estimated Total Cost: ${estimated_cost:.4f} (based on example pricing)")

Q3:
input tokens: 2399
output tokens: 153
Estimated Total Cost: $0.0018 (based on example pricing)


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

In [22]:
from gitsource import chunk_documents

# Chunk the documents into smaller pieces with a specified size and step for overlapping.
chunks = chunk_documents(documents, size=2000, step=1000)

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

In [23]:
# Print the total number of chunks created.
print(f"Q4: {len(chunks)}")

Q4: 295


## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

# алгоритм дій:
1. Розбиваємо документи на chunks.
2. Індексуємо chunks замість цілих документів.
3. Пошук повертає лише ті chunks, де є релевантний текст.
4. Формуємо prompt із цих chunks.

In [24]:
import minsearch
# Initialize a new minsearch.Index for the chunks, with 'content' as text and 'filename' as keyword.
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
# Fit the index with the generated chunks.
chunk_index.fit(chunks)

In [25]:
import time
from sqlitesearch import TextSearchIndex

# Initialize a new TextSearchIndex for chunks
chunk_index_sql = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lesson_llm_course_chunks.db" # Using a new database file for chunks
)

# Add each chunk to the new SQL index
for i, chunk in enumerate(chunks):
    chunk_index_sql.add(chunk)
    # print(f"Added chunk {i+1} from {chunk['filename']}...")
    # time.sleep(0.01) # Small delay to avoid overwhelming the system if many chunks

chunk_index_sql.close()
print("Done. Chunk index saved to lesson_llm_course_chunks.db")

Done. Chunk index saved to lesson_llm_course_chunks.db


In [26]:
import os

file_path_chunks = 'lesson_llm_course_chunks.db'
file_size_mb_chunks = os.path.getsize(file_path_chunks)/ 1024 ** 2

print(f"File size of {file_path_chunks}: {file_size_mb_chunks:.2f} MB)")

File size of lesson_llm_course_chunks.db: 1.94 MB)


In [27]:
# Initialize RAGBase with the chunked index, Groq client, and specified model.
rag_helper = RAGBase(
    index=chunk_index_sql,
    llm_client=client_groq,
    model="llama-3.3-70b-versatile",
    # model="openai/gpt-oss-120b"
)
# Run the RAG query with the chunked data.
result = rag_helper.rag(query)
# Print the answer from the RAG result.
print(result.answer)
# Store the input tokens for Q5.
Q5_input_tokens = result.input_tokens
# Print the input tokens used.
print("Input tokens:", Q5_input_tokens)
# Print the output tokens used.
print("Output tokens:", result.output_tokens)

estimated_cost = calculate_estimated_cost(Q5_input_tokens, result.output_tokens)
print(f"Estimated Total Cost: ${estimated_cost:.4f} (based on example pricing)")

The agentic loop keeps calling the model until it stops by using a `while` loop that continues to execute indefinitely (`while True`) until a certain condition is met. 

The condition that stops the loop is when the model returns a response without any function calls (`has_function_calls == False`). This is checked after each iteration, and if no function calls are found, the loop breaks (`break`). 

In each iteration, the loop calls the model with the current input, checks the response for function calls, and if any are found, it executes them and adds the output to the input for the next iteration. This process repeats until the model returns a response without any function calls, at which point the loop stops.
Input tokens: 575
Output tokens: 145
Estimated Total Cost: $0.0005 (based on example pricing)


In [28]:
# Calculate and print how many times fewer input tokens were used in Q5 compared to Q3.
print(f"Q5: {int(Q3_input_tokens/Q5_input_tokens)}x fewer")

Q5: 4x fewer


## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

In [29]:
import json

# Initialize a counter for search calls.
search_call_count = 0

# Define the search function, which simulates searching the course lessons.
# It increments a global counter and uses chunk_index to perform the actual search.
def search(query: str):
    """Search the course lessons for relevant content."""
    global search_call_count
    search_call_count += 1

    results = chunk_index_sql.search(query, num_results=5)
    return results

# Define available functions mapping their names to the actual function objects.
available_functions = {"search": search}

# Helper function to execute a function call (currently only 'search').
# It parses arguments, calls the function, and formats the output.
def make_call(tool_call, available_functions):
    """
    Выполняет функцию, запрошенную моделью, и возвращает сообщение
    в формате, который понимает Groq API.

    :param tool_call: Объект ChatCompletionMessageToolCall из ответа LLM
    :param available_functions: Словарь вида {'имя_функции': объект_функции_python}
    """
    function_name = tool_call.function.name
    # Аргументы приходят в виде строки JSON, их нужно распарсить
    function_args = json.loads(tool_call.function.arguments)

    print(f"Выполняю функцию: {function_name} с аргументами {function_args}")

    # Поиск функции в словаре доступных инструментов
    function_to_call = available_functions.get(function_name)

    if function_to_call:
        try:
            # Вызываем функцию
            function_response = function_to_call(**function_args)
        except Exception as e:
            function_response = f"Ошибка при выполнении функции: {str(e)}"
    else:
        function_response = f"Ошибка: Функция {function_name} не найдена."

    # Groq API требует, чтобы результат был строкой
    content = json.dumps(function_response, ensure_ascii=False)

    # Возвращаем словарь в формате сообщения 'tool'
    return {
        "role": "tool",
        "tool_call_id": tool_call.id, # ID, который прислала LLM для этого вызова
        "name": function_name,
        "content": content
    }

# Define the schema for the 'search' tool, describing its type, name, description, and parameters.
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course lessons knowledge base for relevant content.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query string to find relevant course lessons."
                }
            },
            "required": ["query"]
        }
    }
}


# Define the instructions for the developer role of the agent.
instructions = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

# Define the user's question for the agent.
question = "How does the agentic loop work, and how is it different from plain RAG?"

# Initialize the list of messages, starting with system instructions and the user's question.
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

# Reset search call count and iteration counter.
search_call_count = 0
it = 1
last_answer = ""

# Initialize token usage counters.
total_input_tokens = 0
total_output_tokens = 0

# Start the agentic loop.
while True:
    print(f"iteration #{it}...")

    # Send messages to the LLM with the search tool available.
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=[search_tool],
        tool_choice="auto"
    )

    # Accumulate token usage from the LLM response.
    total_input_tokens += response.usage.prompt_tokens
    total_output_tokens += response.usage.completion_tokens

    # Get the LLM's message (assistant's response)
    llm_message = response.choices[0].message

    # Manually construct the dictionary for the assistant's message
    # to ensure only supported fields are included for the next API call.
    assistant_message_dict = {
        "role": llm_message.role,
        "content": llm_message.content,
    }
    if llm_message.tool_calls:
        # Convert Pydantic tool_calls objects to dictionaries for clean append
        # using model_dump(exclude_unset=True) to only include explicitly set fields.
        assistant_message_dict["tool_calls"] = [
            tool_call.model_dump(exclude_unset=True) for tool_call in llm_message.tool_calls
        ]
    messages.append(assistant_message_dict)

    has_function_calls = False # Reset for this iteration

    # Check if the LLM's message includes tool calls
    if llm_message.tool_calls:
        has_function_calls = True
        print(f"LLM requested tool calls: {len(llm_message.tool_calls)}")
        for tool_call in llm_message.tool_calls:
            # Execute the tool call
            tool_message = make_call(tool_call, available_functions)
            # Add the tool's output to the conversation history
            messages.append(tool_message)
    elif llm_message.content:
        # If no tool calls, but content is present, this is the final answer
        last_answer = llm_message.content
        print("ASSISTANT:")
        print(last_answer)
        # No function calls, so has_function_calls remains False
    else:
        print("No tool calls and no content in LLM message.")

    print(messages) # Debugging output

    # Increment iteration counter.
    it = it + 1
    # Break the loop if no function calls were made in this iteration,
    # meaning the LLM provided a final textual answer or no action.
    if not has_function_calls:
        print("No function calls detected or final answer provided, breaking loop.")
        break

print(f"\n--- Number of Search Calls ---")
print(f"The agent called search {search_call_count} times")

estimated_cost = calculate_estimated_cost(total_input_tokens, total_output_tokens)

print(f"\n--- Token Usage and Cost ---")
print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")
print(f"Estimated Total Cost: ${estimated_cost:.4f} (based on example pricing)")

iteration #1...
LLM requested tool calls: 3
Выполняю функцию: search с аргументами {'query': 'agentic loop vs plain RAG'}
Выполняю функцию: search с аргументами {'query': 'agentic loop explanation'}
Выполняю функцию: search с аргументами {'query': 'RAG and agentic loop differences'}
[{'role': 'system', 'content': "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."}, {'role': 'user', 'content': 'How does the agentic loop work, and how is it different from plain RAG?'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': '4s66mykeq', 'function': {'arguments': '{"query":"agentic loop vs plain RAG"}', 'name': 'search'}, 'type': 'function'}, {'id': 'sm4pht0n2', 'function': {'arguments': '{"query":"agentic loop explanation"}', 'name': 'search'}, 'type': 'function'}, {'id': 'fv4a53fda', 'function': {'arguments': '{"query":"RAG and agentic loop differences"}', 'name': 'search'}, 'ty

In [ ]:
# Install the toyaikit library, a small agent library for learning and experimentation.
%pip install toyaikit

In [31]:
from toyaikit.tools import Tools

# Initialize ToyAIKit's Tools manager.
agent_tools = Tools()
# Register the 'search' function and its corresponding tool schema with agent_tools.
agent_tools.add_tool(search, search_tool)

print("agent_tools initialized and search function registered.")

agent_tools initialized and search function registered.


In [32]:
# Display the tool schema that ToyAIKit generated or is using for the registered tools.
agent_tools.get_tools()

[{'type': 'function',
  'function': {'name': 'search',
   'description': 'Search the course lessons knowledge base for relevant content.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query string to find relevant course lessons.'}},
    'required': ['query']}}}]

In [43]:
import json
import uuid
from toyaikit.llm import LLMClient, Message
from toyaikit.types import TextBlock # Attempting import from toyaikit.types
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback
from typing import List, Optional, Union
from groq import Groq

from pydantic import BaseModel, Field

# Import OpenAI Pydantic models for compatibility (still useful for internal parsing from Groq)
from openai.types.chat.chat_completion import ChatCompletion, Choice
from openai.types.completion_usage import CompletionUsage as OpenAICompletionUsage
from openai.types.chat.chat_completion_message import ChatCompletionMessage
from openai.types.chat.chat_completion_message_tool_call import ChatCompletionMessageToolCall, Function as OpenAI_Function
from toyaikit.chat.runners import BaseToolUsingRunner, RunnerCallback, LoopResult
from toyaikit.pricing import CostInfo, PricingConfig, TokenUsage
from toyaikit.tools import Tools
from openai.types.responses.response_function_tool_call import ResponseFunctionToolCall
from openai.types.responses.easy_input_message import EasyInputMessage

# Removed ToyAIKitUsage class as we'll use TokenUsage directly
# Removed ToyAIKitContentPart class as we'll use TextBlock directly

# Assuming LLMClient and Tools are already defined in a previous cell and are in scope.

class GroqClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
        api_key: Optional[str] = None,
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqClient"
            )

        self.model = model

        if client is None:
            self.client = Groq(api_key=api_key)
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List, # toyaikit passes this as a list of dictionaries, or EasyInputMessage/Message objects
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        # Convert toyaikit's messages to Groq-compatible message format
        groq_messages = []
        for msg in chat_messages:
            if isinstance(msg, dict):
                # If it's a raw dict (e.g., tool output from agent), append as is.
                groq_messages.append(msg)
            elif isinstance(msg, EasyInputMessage): # Handle EasyInputMessage for user/system input
                groq_messages.append({"role": msg.role, "content": msg.content})
            elif hasattr(msg, 'model_dump'): # if it's a pydantic model like toyaikit.llm.Message
                msg_dict = msg.model_dump(exclude_unset=True)
                if 'tool_calls' in msg_dict and msg_dict['tool_calls']:
                    groq_tool_calls = []
                    for tc in msg_dict['tool_calls']:
                        if isinstance(tc, ChatCompletionMessageToolCall):
                            groq_tool_calls.append({
                                "id": tc.id,
                                "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                                "type": "function"
                            })
                        else:
                            groq_tool_calls.append({
                                "id": tc.get('id', str(uuid.uuid4())),
                                "function": {"name": tc['function']['name'], "arguments": tc['function']['arguments']},
                                "type": "function"
                            })
                    msg_dict['tool_calls'] = groq_tool_calls
                    msg_dict['content'] = None

                if 'content' in msg_dict and msg_dict['content'] is not None:
                    if isinstance(msg_dict['content'], list):
                        # Assuming content is List[TextBlock], extract text
                        msg_dict['content'] = " ".join([part.text for part in msg_dict['content'] if hasattr(part, 'text')])
                    # No need for ToyAIKitContentPart specific handling anymore

                groq_messages.append(msg_dict)
            else:
                # Fallback for unexpected message types
                groq_messages.append(dict(msg))

        args = dict(
            model=self.model,
            messages=groq_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        if 'output_format' in kwargs:
            kwargs.pop('output_format')

        args.update(kwargs)

        groq_raw_response = self.client.chat.completions.create(**args)

        # --- Start of adaptation to toyaikit.llm.Response-compatible object ---

        # 1. Adapt usage object to TokenUsage
        toyaikit_usage = None
        if hasattr(groq_raw_response, 'usage') and groq_raw_response.usage:
            groq_usage = groq_raw_response.usage
            toyaikit_usage = TokenUsage(
                model=self.model, # Add model to TokenUsage
                input_tokens=groq_usage.prompt_tokens,
                output_tokens=groq_usage.completion_tokens,
            )

        # 2. Adapt choices/messages to a list of toyaikit.llm.Message objects
        toyaikit_messages_output = []
        if groq_raw_response.choices:
            groq_choice = groq_raw_response.choices[0]
            groq_message = groq_choice.message

            # Prepare content for toyaikit.llm.Message
            message_content = []
            if groq_message.content:
                message_content.append(TextBlock(text=groq_message.content))

            # Handle tool calls first (if any)
            if groq_message.tool_calls:
                toyaikit_tool_calls_list = []
                for tc in groq_message.tool_calls: # tc is groq.types.chat.chat_completion_message_tool_call.ChatCompletionMessageToolCall
                    toyaikit_tool_calls_list.append(
                        ChatCompletionMessageToolCall(
                            id=tc.id,
                            function=OpenAI_Function(
                                name=tc.function.name,
                                arguments=tc.function.arguments
                            ),
                            type="function"
                        )
                    )

                toyaikit_messages_output.append(
                    Message(
                        id=str(uuid.uuid4()),
                        role="assistant",
                        content=message_content, # Use TextBlock wrapped content
                        tool_calls=toyaikit_tool_calls_list,
                        type="message",
                        model=self.model, # Ensure model is present
                        usage=toyaikit_usage # Pass TokenUsage object directly
                    )
                )

            elif groq_message.content: # If only content and no tool calls
                toyaikit_messages_output.append(
                    Message(
                        id=str(uuid.uuid4()),
                        role=groq_message.role,
                        content=message_content, # Use TextBlock wrapped content
                        type="message",
                        model=self.model, # Ensure model is present
                        usage=toyaikit_usage # Pass TokenUsage object directly
                    )
                )

        # 3. Construct a toyaikit-compatible response object
        class ToyAIKitCompatibleResponse(BaseModel):
            output: List[Message]
            usage: Optional[TokenUsage] = None # Use TokenUsage here

        return ToyAIKitCompatibleResponse(output=toyaikit_messages_output, usage=toyaikit_usage)

    @property
    def responses(self):
        """Compatibility layer for RAGBase and other tools expecting .responses.create"""
        return self

    def create(self, model: str = None, input: List = None, **kwargs):
        """Compatibility method for the 'responses' API."""
        return self.send_request(
            chat_messages=input,
            model=model or self.model,
            **kwargs
        )

# New GroqResponsesRunner class
class GroqResponsesRunner(BaseToolUsingRunner):
    """Runner for Groq chat completions API, compatible with toyaikit's BaseToolUsingRunner."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            # Groq prefers 'system' role for initial instructions. Using EasyInputMessage.
            return [
                EasyInputMessage(
                    role="system",
                    content=self.developer_prompt,
                )
            ]
        return list(previous_messages)

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)
        else:
            # Initialize with system prompt using EasyInputMessage
            chat_messages.append(
                EasyInputMessage(
                    role="system",
                    content=self.developer_prompt,
                )
            )

        # Add user's message using EasyInputMessage
        user_message = EasyInputMessage(
            role="user",
            content=prompt,
        )
        chat_messages.append(user_message)

        total_input_tokens = 0
        total_output_tokens = 0
        last_message_text = ""

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )

            if callback:
                callback.on_response(response)

            if response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            has_function_calls = False

            if not response.output:
                break

            for msg_item in response.output: # msg_item is toyaikit.llm.Message
                chat_messages.append(msg_item)

                if msg_item.tool_calls:
                    has_function_calls = True
                    for tc in msg_item.tool_calls:
                        function_call_for_tool = ResponseFunctionToolCall(
                            type="function_call",
                            name=tc.function.name,
                            arguments=tc.function.arguments,
                            call_id=tc.id,
                        )

                        tool_result_dict = self.tools.function_call(function_call_for_tool)

                        tool_output_message = Message(
                            id=str(uuid.uuid4()),
                            role="tool",
                            tool_call_id=tc.id,
                            content=[TextBlock(text=tool_result_dict['output'])], # Use TextBlock
                            type="tool_code",
                            model=self.llm_client.model
                        )
                        chat_messages.append(tool_output_message)

                        if callback:
                            callback.on_function_call(function_call_for_tool, tool_result_dict['output'])

                elif msg_item.content:
                    if callback:
                        # msg_item.content is List[TextBlock]
                        content_text = " ".join([cp.text for cp in msg_item.content])
                        callback.on_message(content_text)
                    last_message_text = " ".join([cp.text for cp in msg_item.content])

            if not has_function_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        final_last_message = None
        if last_message_text:
            if output_format:
                final_last_message = output_format.model_validate_json(last_message_text)
            else:
                final_last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=final_last_message,
        )

# Initialize an IPythonChatInterface for interactive chat display in the notebook.
chat_interface = IPythonChatInterface()
# Initialize a DisplayingRunnerCallback to visualize model messages and tool calls.
callback = DisplayingRunnerCallback(chat_interface)

# Initialize an OpenAIClient for ToyAIKit, using the same model as in the agentic loop
# and explicitly passing the Groq API key for authentication.
llm_client_toyaikit = GroqClient(model="llama-3.3-70b-versatile", api_key=api_key_groq) # Pass the api_key_groq here

# Initialize the GroqResponsesRunner, which orchestrates the agentic loop.
# It takes registered tools, developer instructions, the chat interface, and the LLM client.
runner = GroqResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client_toyaikit
)

print("Chat interface and runner initialized.")

ModuleNotFoundError: No module named 'toyaikit.content'

In [34]:
# Execute the agentic loop using the runner with the defined question and callback.
# The 'question' variable is taken from Q6's definition.
result = runner.loop(
    prompt=question, # Using the 'question' from Q6
    callback=callback,
)

ValidationError: 15 validation errors for Message
content.0.TextBlock
  Input should be a valid dictionary or instance of TextBlock [type=model_type, input_value=ToyAIKitContentPart(text=...ords before answering."), input_type=ToyAIKitContentPart]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
content.0.ThinkingBlock
  Input should be a valid dictionary or instance of ThinkingBlock [type=model_type, input_value=ToyAIKitContentPart(text=...ords before answering."), input_type=ToyAIKitContentPart]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
content.0.RedactedThinkingBlock
  Input should be a valid dictionary or instance of RedactedThinkingBlock [type=model_type, input_value=ToyAIKitContentPart(text=...ords before answering."), input_type=ToyAIKitContentPart]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type
content.0.ToolUseBlock
  Input should be a valid dictionary or instance of ToolUseBl

In [ ]:
# Print the total number of times the 'search' function was explicitly called by the agent.
print(f"The agent called search {search_call_count} times.")

In [ ]:
# Display the complete history of messages, including system, user, assistant, function calls, and function call outputs, from the agent's run.
print(result.all_messages)

In [ ]:
print("\n--- Analyzing Tool Calls History ---")
# Iterate through all messages in the result to analyze the structure of tool calls.
for i, message in enumerate(result.all_messages):
    # Check if the message type is a function call.
    if message.type == "function_call":
        print(f"Message {i}: Function Call - Name: {message.name}, Arguments: {message.arguments}")
    # Check if the message type is a function call output.
    elif message.type == "function_call_output":
        print(f"Message {i}: Function Call Output - Call ID: {message.call_id}, Output: {message.output}")
    # Optionally, you can also print assistant messages.
    elif message.type == "message":
        # print(f"Message {i}: Assistant Message - Content: {message.content[0].text}")
        pass

# ToyAIKit

Видео: [Смотреть этот урок](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Написанный вручную агентный цикл из предыдущего урока полезен для обучения, но он довольно однообразен. Каждый раз при создании нового агента вам пришлось бы писать один и тот же цикл `while`, одну и ту же обработку вызовов функций и управление сообщениями.

`ToyAIKit` инкапсулирует этот паттерн, позволяя вам сосредоточиться на инструментах, промптах и поведении. Мы построили его вместе на одном из воркшопов DataTalks.Club некоторое время назад. Он делает то же самое, что и наш ручной цикл, но с меньшим количеством шаблонного кода. Если вы заглянете в код `runners`, вы найдете там тот же цикл `while True`, который мы писали сами.

Я использую его здесь намеренно, потому что не хочу выбирать победителя среди промышленных фреймворков. `ToyAIKit` маленький и легко читается, поэтому, если что-то сломается, вы сможете увидеть, что именно произошло. Это делает его удобным для разработки и отладки локально перед переходом в продакшн.

Одно предостережение: `ToyAIKit` — это библиотека для обучения и экспериментов, она НЕ предназначена для использования в реальных проектах (production). Мы используем ее, потому что она минималистична и наглядна.

## Настройка

Установите библиотеку:

```bash
uv add toyaikit
```

Импортируйте необходимые классы:

```python
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
```

## Регистрация инструмента

Мы регистрируем нашу функцию `search` вместе со схемой из предыдущих уроков:

```python
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
```

## Автоматическая генерация схемы в ToyAIKit

Писать схему вручную утомительно, и мы не хотим делать это для каждой функции. На самом деле, это и не обязательно.

Если мы добавим аннотации типов и строку документации (docstring) к функции `search`, `ToyAIKit` прочитает их и сам составит схему за нас:

```python
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )
```

Затем зарегистрируйте ее без передачи схемы:

```python
agent_tools = Tools()
agent_tools.add_tool(search)
```

Вы можете посмотреть, что сгенерировал `ToyAIKit`:

```python
agent_tools.get_tools()
```

На выходе вы получите ту же JSON-схему, которую мы писали вручную в уроке про вызов функций. `ToyAIKit` создал ее на основе докстринга и аннотации типа.

Любой современный агентный фреймворк использует этот прием. Он считывает типизированную функцию Python с документацией и строит на ее основе схему. Так работают OpenAI Agents SDK, PydanticAI, LangChain и Google ADK. Вы пишете инструмент, а фреймворк сам понимает, как его описать.

## Интерфейс чата и раннер (runner)

Создайте интерфейс чата и колбэк, а затем постройте раннер:

```python
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
```

`chat_interface` отвечает за отображение в ноутбуке. `callback` отрисовывает сообщения модели и вызовы инструментов по мере их возникновения. Раннер запускает агентный цикл — тот самый `while True`, который мы писали вручную. Он отправляет сообщения, выполняет вызовы функций, добавляет результаты инструментов обратно и повторяет процесс до тех пор, пока модель не закончит.

Мы намеренно выбрали здесь `gpt-5.4-mini`. Без нее `ToyAIKit` использует более простую и быструю модель по умолчанию, которая не так надежно следует инструкциям.

## Запуск одного промпта

Запустите один запрос:

```python
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)
```

Мы специально использовали опечатку «Olama». Агент выполняет поиск, получает плохие результаты, а затем пробует еще раз с «Ollama». Механизм восстановления такой же, как в ручном цикле. Вывод в ноутбуке выглядит гораздо приятнее: каждый вызов инструмента и сообщение отображаются по очереди, так что вы можете изучить каждый результат поиска.

`result` — это объект `LoopResult`, содержащий `all_messages` (всю историю диалога), количество токенов и `cost` (стоимость, рассчитанную на основе использования токенов).

## Стоимость и токены

Посмотрите, сколько стоил этот вызов:

```python
result.cost
```

Это полезно при разработке — особенно для многоходовых агентов, где один запрос может вызвать несколько обращений к модели. В ручном цикле вам приходилось считать это самим. Фреймворк делает это за вас автоматически.

Вы также можете просмотреть полную историю сообщений:

```python
result.all_messages
```

Это просто список — тот же список `messages`, который мы вели вручную.

## Продолжение диалога

Возьмите сообщения из предыдущего результата и передайте их как `previous_messages` при следующем вызове `loop`:

```python
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)
```

Раннер продолжит с того места, где остановился предыдущий вызов, используя тот же агентный цикл и расширенную историю. Модель поймет, что под «другой моделью» подразумевается Ollama, так как она видит предыдущий ход в памяти. Без этой истории она бы понятия не имела, о чем идет речь.

## Интерактивный чат

Для работы в режиме чата запустите встроенный цикл ввода:

```python
runner.run()
```

Вводите вопросы и получайте ответы. Введите «stop», чтобы выйти.

[← Агентный цикл](14-agentic-loop.md) | [Другие фреймворки →](16-other-frameworks.md)


In [ ]:
from typing import List, Optional

from openai import OpenAI
from pydantic import BaseModel

from openai.types.chat.chat_completion import ChatCompletion
from openai.types.chat.parsed_chat_completion import ParsedChatCompletion
from openai.types.responses.response import Response
from openai.types.responses.parsed_response import ParsedResponse

from anthropic.types import Message, RawMessageStopEvent

from toyaikit.tools import Tools


class LLMClient:
    def send_request(self, chat_messages: List, tools: Tools = None):
        raise NotImplementedError("Subclasses must implement this method")

class GroqClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
        api_key: Optional[str] = None, # Added api_key parameter
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqClient"
            )

        self.model = model

        if client is None:
            # Pass the api_key directly to the Groq client
            self.client = Groq(api_key=api_key)
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        args = dict(
            model=self.model,
            messages=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        # Remove 'output_format' if present, as Groq API does not support it directly
        # or it is handled differently by the Groq client. ToyAIKit passes it by default.
        if 'output_format' in kwargs:
            kwargs.pop('output_format')

        args.update(kwargs)

        return self.client.chat.completions.create(**args)

    @property
    def responses(self):
        """Compatibility layer for RAGBase and other tools expecting .responses.create"""
        return self

    def create(self, model: str = None, input: List = None, **kwargs):
        """Compatibility method for the 'responses' API."""
        return self.send_request(
            chat_messages=input,
            model=model or self.model,
            **kwargs
        )

class GroqChatCompletionsClient(LLMClient):
    def __init__(
        self,
        model: str = "openai/gpt-oss-120b",
        client=None,
        extra_kwargs: dict = None,
    ):
        try:
            from groq import Groq
        except ImportError:
            raise ImportError(
                "Please run 'pip install groq' to use GroqChatCompletionsClient"
            )

        self.model = model

        if client is None:
            self.client = Groq()
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def convert_single_tool(self, tool):
        """
        Convert a single OpenAI tool/function API dict to Groq-compatible format.
        """
        return {
            "type": "function",
            "function": {
                "name": tool["name"],
                "description": tool["description"],
                "parameters": tool["parameters"],
            },
        }

    def convert_api_tools_to_chat_functions(self, api_tools):
        """
        Convert a list of OpenAI API tools to Groq-compatible format.
        """
        chat_functions = []

        for tool in api_tools:
            converted = self.convert_single_tool(tool)
            chat_functions.append(converted)

        return chat_functions

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        **kwargs,
    ):
        tools_list = []

        if tools is not None:
            tools_requests_format = tools.get_tools()
            tools_list = self.convert_api_tools_to_chat_functions(
                tools_requests_format,
            )

        args = dict(
            model=self.model,
            messages=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )
        args.update(kwargs)

        return self.client.chat.completions.create(**args)

In [ ]:
import json
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Callable, Generic, TypeVar

from openai.types.chat.chat_completion_function_message_param import (
    ChatCompletionFunctionMessageParam,
)
from openai.types.chat.chat_completion_system_message_param import (
    ChatCompletionSystemMessageParam,
)
from openai.types.chat.chat_completion_user_message_param import (
    ChatCompletionUserMessageParam,
)
from openai.types.responses.easy_input_message import EasyInputMessage
from openai.types.responses.response_function_tool_call import ResponseFunctionToolCall
from pydantic import BaseModel

from toyaikit.chat.interface import ChatInterface
from toyaikit.llm import LLMClient
from toyaikit.pricing import CostInfo, PricingConfig, TokenUsage
from toyaikit.tools import Tools

# T must be either a str or a (subclass)
# instance of pydantic BaseModel
T = TypeVar("T", str, BaseModel)


def _get_tool_call_output(call_result) -> str:
    """Extract output from tool call result, handling both dict and object types."""
    if isinstance(call_result, dict):
        return call_result.get("output", "")
    return getattr(call_result, "output", "")


@dataclass
class LoopResult(Generic[T]):
    new_messages: list
    all_messages: list
    tokens: TokenUsage
    cost: CostInfo | None
    last_message: T


class RunnerCallback(ABC):
    """Abstract base class for different chat runners."""

    @abstractmethod
    def on_function_call(self, function_call: dict, result: str):
        """
        Called when a function call is made.
        """
        pass

    @abstractmethod
    def on_message(self, message: dict):
        """
        Called when a message is received.
        """
        pass

    @abstractmethod
    def on_reasoning(self, reasoning: str):
        """
        Called when a reasoning is received.
        """
        pass

    @abstractmethod
    def on_response(self, response):
        pass


class ChatRunner(ABC):
    """Abstract base class for different chat runners."""

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_type: BaseModel = None
    ) -> LoopResult:
        """
        Do one tool-call loop.
        """
        pass

    @abstractmethod
    def run(self, previous_messages: list = None) -> LoopResult:
        """
        Repeast tool call loops until user asks to stop
        """
        pass


class DisplayingRunnerCallback(RunnerCallback):
    def __init__(self, chat_interface: ChatInterface):
        self.chat_interface = chat_interface

    def on_function_call(self, function_call, result):
        self.chat_interface.display_function_call(
            function_call.name,
            function_call.arguments,
            result,
        )

    def on_message(self, message):
        self.chat_interface.display_response(message)

    def on_reasoning(self, reasoning):
        self.chat_interface.display_reasoning(reasoning)

    def on_response(self, response):
        self.chat_interface.display("-> Response received")


class BaseToolUsingRunner(ChatRunner):
    """Base class for runners that use tools and LLM clients.

    Provides common initialization and run() method implementation.
    Subclasses only need to implement the loop() method.
    """

    def __init__(
        self,
        tools: Tools = None,
        developer_prompt: str = "You're a helpful assistant.",
        chat_interface: ChatInterface = None,
        llm_client: LLMClient = None,
        pricing_config: PricingConfig = None,
    ):
        self.tools = tools
        self.developer_prompt = developer_prompt
        self.chat_interface = chat_interface
        self.llm_client = llm_client
        self.displaying_callback = DisplayingRunnerCallback(chat_interface)
        self.pricing_config = pricing_config or PricingConfig()

    @abstractmethod
    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        """Execute one tool-call loop. Must be implemented by subclasses."""
        pass

    def run(
        self,
        previous_messages: list = None,
        stop_criteria: Callable = None,
    ) -> LoopResult:
        """Repeat tool-call loops until user asks to stop."""
        chat_messages = self._initialize_messages(previous_messages)

        total_input_tokens = 0
        total_output_tokens = 0
        last_message_text = ""

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            loop_result = self.loop(
                prompt=user_input,
                previous_messages=chat_messages,
                callback=self.displaying_callback,
            )

            chat_messages.extend(loop_result.new_messages)
            total_input_tokens += loop_result.tokens.input_tokens
            total_output_tokens += loop_result.tokens.output_tokens
            last_message_text = loop_result.last_message

            if stop_criteria and stop_criteria(loop_result.new_messages):
                break

        combined_cost = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )
        combined_tokens = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        return LoopResult(
            new_messages=chat_messages,
            all_messages=chat_messages,
            tokens=combined_tokens,
            cost=combined_cost,
            last_message=last_message_text,
        )

    @abstractmethod
    def _initialize_messages(self, previous_messages: list = None) -> list:
        """Initialize chat messages. Must be implemented by subclasses."""
        pass


class OpenAIResponsesRunner(BaseToolUsingRunner):
    """Runner for OpenAI responses API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [
                EasyInputMessage(
                    role="developer",
                    content=self.developer_prompt,
                )
            ]
        return list(previous_messages)  # Return a copy

    def loop(
        self,
        prompt: str,
        previous_messages: list[dict] = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append(
                EasyInputMessage(
                    role="developer",
                    content=self.developer_prompt,
                )
            )
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        chat_messages.append(
            EasyInputMessage(
                role="user",
                content=prompt,
            )
        )

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )
            if callback:
                callback.on_response(response)

            if hasattr(response, "usage") and response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            has_function_calls = False

            chat_messages.extend(response.output)

            for entry in response.output:
                if entry.type == "function_call":
                    result = self.tools.function_call(entry)
                    chat_messages.append(result)
                    if callback:
                        callback.on_function_call(entry, result['output'])
                    has_function_calls = True

                elif entry.type == "message":
                    if callback:
                        callback.on_message(entry.content[0].text)

            if not has_function_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        last_message_text = ""
        last_message = None
        for entry in reversed(response.output):
            if entry.type == "message":
                last_message_text = entry.content[0].text
                if output_format:
                    last_message = output_format.model_validate_json(last_message_text)
                else:
                    last_message = last_message_text
                break

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=last_message,
        )


class OpenAIAgentsSDKRunner(ChatRunner):
    """Runner for OpenAI Agents SDK."""

    def __init__(self, chat_interface: ChatInterface, agent):
        try:
            from agents import Runner
        except ImportError:
            raise ImportError(
                "Please run 'pip install openai-agents' to use this feature"
            )

        self.agent = agent
        self.runner = Runner()
        self.chat_interface = chat_interface

    async def run(self) -> None:
        from agents import SQLiteSession

        session_id = f"chat_session_{uuid.uuid4().hex[:8]}"
        session = SQLiteSession(session_id)

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            result = await self.runner.run(
                self.agent, input=user_input, session=session
            )

            func_calls = {}
            for ni in result.new_items:
                raw = ni.raw_item
                if ni.type == "tool_call_item":
                    func_calls[raw.call_id] = raw

            for ni in result.new_items:
                raw = ni.raw_item

                if ni.type == "handoff_call_item":
                    raw = ni.raw_item
                    self.chat_interface.display(f"handoff: {raw.name}")

                if ni.type == "handoff_output_item":
                    self.chat_interface.display(
                        f"handoff: {ni.target_agent.name} -> {ni.source_agent.name} successful"
                    )

                if ni.type == "tool_call_output_item":
                    call_id = raw["call_id"]
                    if call_id not in func_calls:
                        self.chat_interface.display(
                            f"error: cannot find the call parameters for {call_id=}"
                        )
                    else:
                        func_call = func_calls[call_id]
                        self.chat_interface.display_function_call(
                            func_call.name, func_call.arguments, raw["output"]
                        )

                if ni.type == "message_output_item":
                    md = raw.content[0].text
                    self.chat_interface.display_response(md)


class PydanticAIRunner(ChatRunner):
    """Runner for Pydantic AI."""

    def __init__(self, chat_interface: ChatInterface, agent):
        self.chat_interface = chat_interface
        self.agent = agent

    async def run(self) -> None:
        message_history = []

        while True:
            user_input = self.chat_interface.input()
            if user_input.lower() == "stop":
                self.chat_interface.display("Chat ended.")
                break

            result = await self.agent.run(
                user_prompt=user_input,
                message_history=message_history
            )

            messages = result.new_messages()
            tool_calls = {}

            for m in messages:
                for part in m.parts:
                    kind = part.part_kind

                    if kind == "text":
                        self.chat_interface.display_response(part.content)

                    elif kind == "tool-call":
                        tool_calls[part.tool_call_id] = part

                    elif kind == "tool-return":
                        call = tool_calls.get(part.tool_call_id)

                        raw_result = part.content
                        if raw_result is None:
                            result_str = ""
                        elif isinstance(raw_result, str):
                            result_str = raw_result
                        else:
                            result_str = json.dumps(raw_result, indent=2, default=str)

                        self.chat_interface.display_function_call(
                            call.tool_name,
                            json.dumps(call.args),
                            result_str
                        )

            message_history.extend(messages)



class OpenAIChatCompletionsRunner(BaseToolUsingRunner):
    """Runner for OpenAI chat completions API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [
                ChatCompletionSystemMessageParam(
                    role="system",
                    content=self.developer_prompt
                )
            ]
        return list(previous_messages)  # Return a copy

    @staticmethod
    def convert_function_output_to_tool_message(data):
        return ChatCompletionFunctionMessageParam(
            role="tool",
            tool_call_id=data["call_id"],
            content=data["output"],
        )

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append(
                ChatCompletionSystemMessageParam(
                    role="system",
                    content=self.developer_prompt
                )
            )
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        user_message = ChatCompletionUserMessageParam(
            role="user",
            content=prompt
        )
        chat_messages.append(user_message)

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            reponse = self.llm_client.send_request(
                chat_messages, self.tools, output_format
            )

            if callback:
                callback.on_response(reponse)

            if reponse.usage:
                total_input_tokens += reponse.usage.prompt_tokens
                total_output_tokens += reponse.usage.completion_tokens

            first_choice = reponse.choices[0]
            message_response = first_choice.message
            chat_messages.append(message_response)

            if hasattr(message_response, "reasoning_content"):
                reasoning = (message_response.reasoning_content or "").strip()
                if reasoning != "" and callback:
                    callback.on_reasoning(reasoning)

            content = (message_response.content or "").strip()
            if content != "" and callback:
                callback.on_message(content)

            calls = []

            if hasattr(message_response, "tool_calls"):
                calls = message_response.tool_calls

            if calls is None:
                break

            if len(calls) == 0:
                break

            for call in calls:
                function_call = ResponseFunctionToolCall(
                    type="function_call",
                    name=call.function.name,
                    arguments=call.function.arguments,
                    call_id=call.id,
                )

                call_result = self.tools.function_call(function_call)
                call_result = self.convert_function_output_to_tool_message(call_result)

                chat_messages.append(call_result)

                if callback:
                    content_val = getattr(call_result, "content", None)
                    if content_val is None and isinstance(call_result, dict):
                        content_val = call_result.get("content")
                    callback.on_function_call(function_call, content_val)

        cost = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        last_message_text = (message_response.content or "").strip()
        if output_format:
            last_message = output_format.model_validate_json(last_message_text)
        else:
            last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost,
            last_message=last_message,
        )


class AnthropicMessagesRunner(BaseToolUsingRunner):
    """Runner for Anthropic Messages API."""

    def _initialize_messages(self, previous_messages: list = None) -> list:
        if previous_messages is None or len(previous_messages) == 0:
            return [{
                "role": "system",
                "content": self.developer_prompt
            }]
        return list(previous_messages)  # Return a copy

    def loop(
        self,
        prompt: str,
        previous_messages: list = None,
        callback: RunnerCallback = None,
        output_format: BaseModel = None,
    ) -> LoopResult:
        chat_messages = []
        prev_messages_len = 0

        if previous_messages is None or len(previous_messages) == 0:
            chat_messages.append({
                "role": "system",
                "content": self.developer_prompt
            })
        else:
            chat_messages.extend(previous_messages)
            prev_messages_len = len(previous_messages)

        chat_messages.append({
            "role": "user",
            "content": prompt
        })

        total_input_tokens = 0
        total_output_tokens = 0

        while True:
            response = self.llm_client.send_request(
                chat_messages=chat_messages,
                tools=self.tools,
                output_format=output_format,
            )

            if callback:
                callback.on_response(response)

            # Track token usage
            if hasattr(response, "usage") and response.usage:
                total_input_tokens += response.usage.input_tokens
                total_output_tokens += response.usage.output_tokens

            # Process the response
            assistant_message = {
                "role": "assistant",
                "content": response.content
            }
            chat_messages.append(assistant_message)

            has_tool_calls = False
            text_content = []

            for block in response.content:
                if block.type == "text":
                    text_content.append(block.text)
                    if callback:
                        callback.on_message(block.text)

                elif block.type == "tool_use":
                    has_tool_calls = True
                    function_call = ResponseFunctionToolCall(
                        type="function_call",
                        name=block.name,
                        arguments=json.dumps(block.input),
                        call_id=block.id,
                    )

                    call_result = self.tools.function_call(function_call)
                    result_output = _get_tool_call_output(call_result)

                    # Anthropic expects tool results in a user message with tool_result blocks
                    tool_result_message = {
                        "role": "tool",
                        "tool_call_id": block.id,
                        "content": result_output
                    }
                    chat_messages.append(tool_result_message)

                    if callback:
                        callback.on_function_call(function_call, result_output)

            if not has_tool_calls:
                break

        cost_info = self.pricing_config.calculate_cost(
            self.llm_client.model, total_input_tokens, total_output_tokens
        )

        token_usage = TokenUsage(
            model=self.llm_client.model,
            input_tokens=total_input_tokens,
            output_tokens=total_output_tokens,
        )

        new_messages = chat_messages[prev_messages_len:]

        # Extract last message text
        last_message_text = ""
        last_message = None
        if text_content:
            last_message_text = "".join(text_content)
            if output_format:
                last_message = output_format.model_validate_json(last_message_text)
            else:
                last_message = last_message_text

        return LoopResult(
            new_messages=new_messages,
            all_messages=chat_messages,
            tokens=token_usage,
            cost=cost_info,
            last_message=last_message,
        )

In [ ]:
🚀 Module 1 of LLM Zoomcamp by @DataTalksClub complete!

Just finished Module 1 - Agentic RAG. Learned how to:

✅ Build a RAG system from scratch in plain Python
✅ Index and search documents with minsearch
✅ Chunk long documents for better retrieval
✅ Turn the RAG pipeline into an agent with function calling

Here's my homework solution: <LINK>

Following along with this amazing free course - who else is learning to build with LLMs?

You can sign up here: https://github.com/DataTalksClub/llm-zoomcamp/